<a href="https://colab.research.google.com/github/MunSu2001/File/blob/main/2026_03_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os

In [ ]:
data_dir = "/kaggle/input/datasets/sinmunsu/sentence-classfication"
for file in os.listdir(data_dir):
    print(data_dir + file)

/kaggle/input/datasets/sinmunsu/sentence-classficationdata4.csv
/kaggle/input/datasets/sinmunsu/sentence-classficationdata2.csv
/kaggle/input/datasets/sinmunsu/sentence-classficationdata3.csv
/kaggle/input/datasets/sinmunsu/sentence-classficationdata1.csv


In [ ]:
df= pd.concat([pd.read_csv(data_dir + "/" + file) for file in os.listdir(data_dir)])

In [ ]:
df.shape

(832462, 2)

In [ ]:
df.발화문.apply(lambda x : len(x)).mean()

np.float64(24.403025002943078)

In [ ]:
df[df.발화문.apply(lambda x : len(x) >= 25)].shape

(333990, 2)

In [ ]:
df2 = df[df.발화문.apply(lambda x : len(x) >= 25)].copy()

In [ ]:
df2.shape

(333990, 2)

In [ ]:
from transformers import BertTokenizerFast, BertForSequenceClassification

model_name = "klue/bert-base"

In [ ]:
label2id = dict(zip(df2.카테고리.unique(), range(4)))

In [ ]:
id2label = {v : k for k, v in label2id.items()}

In [ ]:
print(label2id)
print(id2label)

{'디지털가전': 0, '의류': 1, '병원': 2, '음식점': 3}
{0: '디지털가전', 1: '의류', 2: '병원', 3: '음식점'}


In [ ]:
num_labels = len(id2label)

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels = num_labels,
    id2label = id2label,
    label2id = label2id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = model.to(device)

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained(model_name)

In [ ]:
tokenizer("오늘은 화요일입니다.")

{'input_ids': [2, 3822, 2073, 14353, 12190, 18, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [ ]:
from torch.utils.data import Dataset

In [ ]:
class MyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.encodings = self.tokenizer(
            texts,
            truncation=True, padding=True, max_length=max_length
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {key : torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(label2id[self.labels[idx]])
        return item


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df2.head(1)

,발화문,카테고리
8,이게 중간 이렇게 뚫려 있어서 되게 귀여운거 같아요,디지털가전


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df2['발화문'], df2['카테고리'], test_size=0.2, random_state=42, stratify=df2.카테고리)

In [ ]:
y_train.value_counts(normalize=True)

카테고리
디지털가전    0.581103
의류       0.227177
음식점      0.103450
병원       0.088270
Name: proportion, dtype: float64

In [ ]:
y_test.value_counts(normalize=True)

카테고리
디지털가전    0.581110
의류       0.227177
음식점      0.103446
병원       0.088266
Name: proportion, dtype: float64

In [ ]:
a = tokenizer(X_train.tolist())

In [ ]:
train_dataset = MyDataset(X_train.tolist(), y_train.tolist(), tokenizer)
val_dataset = MyDataset(X_test.tolist(), y_test.tolist(), tokenizer)

In [ ]:
a = next(iter(train_dataset))

In [ ]:
a

{'input_ids': tensor([    2,  3651,  2031,  4818,  8913,  1892,   904,  1366,  1295,  1513,
          2259, 30218,  2119,  1513,  2075,  2182,    35,     3,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0]),
 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np

In [ ]:
# 평가지표
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    accuracy = (preds == labels).mean()
    return {"accuracy": float(accuracy)}

In [ ]:
training_args = TrainingArguments(
    output_dir="./klue_bert_sentiment",
    # evaluation_strategy="epoch",
    # save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    # load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    # tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
training_args = TrainingArguments(
        output_dir="./klue_bert_sentiment",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        save_total_limit = 1,
        report_to="none")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.468617,0.206640,0.935178
2,0.188085,0.209076,0.941435
3,0.080752,0.251852,0.943232


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=100197, training_loss=0.17460283716373334, metrics={'train_runtime': 6784.8836, 'train_samples_per_second': 118.141, 'train_steps_per_second': 14.768, 'total_flos': 2.636341181827891e+16, 'train_loss': 0.17460283716373334, 'epoch': 3.0})

In [ ]:
trainer.save_model("./my_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
tokenizer.save_pretrained("./my_model")

('./my_model/tokenizer_config.json', './my_model/tokenizer.json')

In [ ]:
!tar -cvf model.tar /kaggle/working/my_model

tar: Removing leading `/' from member names
/kaggle/working/my_model/
/kaggle/working/my_model/config.json
/kaggle/working/my_model/model.safetensors
/kaggle/working/my_model/training_args.bin
/kaggle/working/my_model/tokenizer_config.json
/kaggle/working/my_model/tokenizer.json


In [ ]:
test =  ['이 음식 상한거 같아요. 맛이 이상해요. ']
encodings = tokenizer(test, max_length=64, return_tensors='pt')
encodings = encodings.to(device)

In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(**encodings)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=-1)
    preds = torch.argmax(probs, dim=-1).tolist()


In [ ]:
id2label[preds[0]]

'음식점'

In [ ]:
def predict(text):
    encodings = tokenizer(text, max_length=64, return_tensors='pt')
    encodings = encodings.to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(**encodings)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1).tolist()
    return id2label[preds[0]]

In [ ]:
predict(["이 옷 수선은 어디에서 해야 하나요?"])

'의류'